# Chapter 12 - Hard Tasks

In these advanced tasks, you'll work with Direct Preference Optimization (DPO) and implement sophisticated evaluation methods for generative models.

---

### Setup

In [ ]:
# Install packages (uncomment if on Colab)
# %%capture
# !pip install -q accelerate==0.31.0 peft==0.11.1 bitsandbytes==0.43.1 transformers==4.41.2 trl==0.9.4 sentencepiece==0.2.0

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, pipeline
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, AutoPeftModelForCausalLM, PeftModel
from trl import SFTTrainer, DPOTrainer, DPOConfig
from datasets import load_dataset, Dataset
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

---

## Task 1: Create Custom Preference Dataset

For DPO training, we need preference pairs (chosen vs rejected responses). In this task, you'll create a custom preference dataset.

**Your task:** 
1. Create a dataset with at least 20 preference pairs
2. Each pair should have: prompt, chosen response, rejected response
3. Format the data properly for DPO training

**Categories to cover:**
- Helpfulness (detailed vs vague responses)
- Harmlessness (safe vs potentially harmful)
- Honesty (accurate vs made-up information)

In [ ]:
# TODO: Create your custom preference dataset
# Template for one example:
# {
#     "prompt": "<|user|>\nYour question here</s>\n<|assistant|>\n",
#     "chosen": "High quality response",
#     "rejected": "Low quality response"
# }

preference_data = [
    # Example 1: Helpfulness
    {
        "prompt": "<|user|>\nWhat is Python?</s>\n<|assistant|>\n",
        "chosen": "Python is a high-level, interpreted programming language created by Guido van Rossum in 1991. It emphasizes code readability with its use of significant indentation. Python supports multiple programming paradigms including procedural, object-oriented, and functional programming. It's widely used for web development, data science, artificial intelligence, automation, and scientific computing. Python has a large standard library and an extensive ecosystem of third-party packages available through PyPI.",
        "rejected": "Python is a programming language. It's used for coding stuff."
    },
    # TODO: Add at least 19 more examples covering different categories
    # YOUR CODE HERE
    {
        "prompt": # YOUR CODE HERE,
        "chosen": # YOUR CODE HERE,
        "rejected": # YOUR CODE HERE
    },
    # Continue adding examples...
]

# Create dataset
custom_dpo_dataset = Dataset.from_list(preference_data)
print(f"Created custom DPO dataset with {len(custom_dpo_dataset)} examples")

In [ ]:
# Verify dataset format
print("Sample from custom dataset:")
print("\nPrompt:", custom_dpo_dataset[0]['prompt'])
print("\nChosen:", custom_dpo_dataset[0]['chosen'])
print("\nRejected:", custom_dpo_dataset[0]['rejected'])

### Questions:

**Q1:** What makes a good "chosen" response compared to a "rejected" one?

*Your answer here*

**Q2:** Why is it important to have diverse preference pairs covering different categories?

*Your answer here*

---

## Task 2: Implement Full SFT + DPO Pipeline

Now you'll implement the complete two-stage fine-tuning pipeline:
1. Stage 1: Supervised Fine-Tuning (SFT)
2. Stage 2: Direct Preference Optimization (DPO)

**Instructions:**
1. First, train a model with SFT on instruction data
2. Then, apply DPO on top using your custom preference dataset
3. Compare the SFT-only model vs SFT+DPO model

### Stage 1: Supervised Fine-Tuning

In [ ]:
# Load and prepare SFT data
template_tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

def format_prompt(example):
    chat = example["messages"]
    prompt = template_tokenizer.apply_chat_template(chat, tokenize=False)
    return {"text": prompt}

# TODO: Load UltraChat dataset for SFT
sft_dataset = (
    load_dataset(# YOUR CODE HERE, split=# YOUR CODE HERE)
      .shuffle(seed=42)
      .select(range(# YOUR CODE HERE))  # Select appropriate size
)
sft_dataset = sft_dataset.map(format_prompt)
print(f"SFT dataset size: {len(sft_dataset)}")

In [ ]:
# TODO: Configure and train SFT model
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=# YOUR CODE HERE,
    bnb_4bit_quant_type=# YOUR CODE HERE,
    bnb_4bit_compute_dtype=# YOUR CODE HERE,
    bnb_4bit_use_double_quant=# YOUR CODE HERE,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_side = "left"

In [ ]:
# TODO: Configure LoRA for SFT
sft_peft_config = LoraConfig(
    lora_alpha=# YOUR CODE HERE,
    lora_dropout=# YOUR CODE HERE,
    r=# YOUR CODE HERE,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj']
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, sft_peft_config)

In [ ]:
# TODO: Configure and run SFT training
sft_training_args = TrainingArguments(
    output_dir="./sft_results",
    per_device_train_batch_size=# YOUR CODE HERE,
    gradient_accumulation_steps=# YOUR CODE HERE,
    optim="paged_adamw_32bit",
    learning_rate=# YOUR CODE HERE,
    lr_scheduler_type="cosine",
    max_steps=# YOUR CODE HERE,  # Adjust based on your dataset size
    logging_steps=10,
    fp16=True,
    gradient_checkpointing=True
)

sft_trainer = SFTTrainer(
    model=model,
    train_dataset=sft_dataset,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=sft_training_args,
    max_seq_length=512,
    peft_config=sft_peft_config,
)

print("Starting SFT training...")
sft_trainer.train()
sft_trainer.model.save_pretrained("tinyllama-sft-qlora")
print("SFT training complete!")

### Stage 2: Direct Preference Optimization

In [ ]:
# TODO: Load the SFT model for DPO training
model = AutoPeftModelForCausalLM.from_pretrained(
    # YOUR CODE HERE,  # Path to SFT model
    low_cpu_mem_usage=True,
    device_map="auto",
    quantization_config=bnb_config,
)
merged_model = model.merge_and_unload()

# Prepare for DPO training
model = prepare_model_for_kbit_training(merged_model)

# TODO: Configure LoRA for DPO
dpo_peft_config = LoraConfig(
    lora_alpha=# YOUR CODE HERE,
    lora_dropout=# YOUR CODE HERE,
    r=# YOUR CODE HERE,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj']
)

model = get_peft_model(model, dpo_peft_config)

In [ ]:
# TODO: Configure and run DPO training
dpo_training_args = DPOConfig(
    output_dir="./dpo_results",
    per_device_train_batch_size=# YOUR CODE HERE,
    gradient_accumulation_steps=# YOUR CODE HERE,
    optim="paged_adamw_32bit",
    learning_rate=# YOUR CODE HERE,  # Lower than SFT (e.g., 1e-5)
    lr_scheduler_type="cosine",
    max_steps=# YOUR CODE HERE,
    logging_steps=5,
    fp16=True,
    gradient_checkpointing=True,
    warmup_ratio=0.1
)

# TODO: Create DPO trainer
dpo_trainer = DPOTrainer(
    model,
    args=dpo_training_args,
    train_dataset=# YOUR CODE HERE,  # Use custom_dpo_dataset
    tokenizer=tokenizer,
    peft_config=dpo_peft_config,
    beta=# YOUR CODE HERE,  # Try 0.1
    max_prompt_length=512,
    max_length=512,
)

print("Starting DPO training...")
dpo_trainer.train()
dpo_trainer.model.save_pretrained("tinyllama-dpo-qlora")
print("DPO training complete!")

### Merge Both Adapters

In [ ]:
# TODO: Load and merge both SFT and DPO adapters
# Step 1: Load SFT model
sft_model = AutoPeftModelForCausalLM.from_pretrained(
    "tinyllama-sft-qlora",
    low_cpu_mem_usage=True,
    device_map="auto",
)
sft_merged = sft_model.merge_and_unload()

# Step 2: Load DPO adapter on top of SFT model
final_model = PeftModel.from_pretrained(
    sft_merged,
    "tinyllama-dpo-qlora",
    device_map="auto",
)
final_model = final_model.merge_and_unload()

print("Successfully merged SFT and DPO adapters!")

### Compare Models

In [ ]:
# TODO: Compare SFT-only vs SFT+DPO on test prompts
test_prompts = [
    "<|user|>\nWhat is the best way to learn programming?</s>\n<|assistant|>\n",
    "<|user|>\nExplain quantum computing in simple terms.</s>\n<|assistant|>\n",
    # Add more test prompts
]

# Load SFT-only model
sft_only = AutoPeftModelForCausalLM.from_pretrained(
    "tinyllama-sft-qlora",
    low_cpu_mem_usage=True,
    device_map="auto",
).merge_and_unload()

# Create pipelines
sft_pipe = pipeline("text-generation", model=sft_only, tokenizer=tokenizer, max_new_tokens=150)
dpo_pipe = pipeline("text-generation", model=final_model, tokenizer=tokenizer, max_new_tokens=150)

# Compare outputs
for prompt in test_prompts:
    print("\n" + "="*80)
    print(f"PROMPT: {prompt.split('<|assistant|>')[0].split('<|user|>')[1].strip()}")
    print("="*80)
    
    print("\nSFT-ONLY MODEL:")
    sft_output = sft_pipe(prompt)[0]['generated_text']
    print(sft_output[len(prompt):])
    
    print("\nSFT+DPO MODEL:")
    dpo_output = dpo_pipe(prompt)[0]['generated_text']
    print(dpo_output[len(prompt):])
    print()

### Questions:

**Q3:** What differences do you notice between SFT-only and SFT+DPO outputs?

*Your answer here*

**Q4:** Does the DPO model seem to follow the preferences you encoded in your training data?

*Your answer here*

**Q5:** What are the limitations of the DPO approach?

*Your answer here*

---

## Task 3: Implement Advanced Evaluation

Evaluating generative models is challenging. In this task, you'll implement multiple evaluation approaches.

**Your task:**
1. Implement perplexity calculation
2. Implement ROUGE score evaluation
3. Create a simple "LLM-as-a-judge" evaluator
4. Perform pairwise comparison

### 3.1 Perplexity Evaluation

In [ ]:
# TODO: Implement perplexity calculation
def calculate_perplexity(model, tokenizer, texts):
    """
    Calculate average perplexity for a list of texts.
    Lower perplexity = better language modeling.
    """
    perplexities = []
    
    for text in texts:
        # YOUR CODE HERE
        # Hint: 
        # 1. Tokenize the text
        # 2. Get model loss with labels=input_ids
        # 3. Perplexity = exp(loss)
        pass
    
    return sum(perplexities) / len(perplexities)

# Test texts
test_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning is a subset of artificial intelligence.",
    "Python is a popular programming language for data science.",
]

# Calculate perplexity for both models
sft_ppl = calculate_perplexity(sft_only, tokenizer, test_texts)
dpo_ppl = calculate_perplexity(final_model, tokenizer, test_texts)

print(f"SFT-only Perplexity: {sft_ppl:.2f}")
print(f"SFT+DPO Perplexity: {dpo_ppl:.2f}")

### 3.2 ROUGE Score Evaluation

In [ ]:
# Install rouge-score if needed
# !pip install rouge-score

from rouge_score import rouge_scorer

# TODO: Create reference answers and evaluate generated responses
eval_examples = [
    {
        "prompt": "<|user|>\nWhat is machine learning?</s>\n<|assistant|>\n",
        "reference": "Machine learning is a branch of artificial intelligence that enables computers to learn from data and improve their performance without being explicitly programmed. It uses algorithms to identify patterns in data and make predictions or decisions."
    },
    # Add more examples
]

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# TODO: Evaluate both models
# Generate responses and calculate ROUGE scores
# YOUR CODE HERE

### 3.3 LLM-as-a-Judge Evaluation

Use a larger model to evaluate the quality of responses from your fine-tuned models.

In [ ]:
# TODO: Implement LLM-as-a-judge evaluation
# If you have access to an API (OpenAI, Anthropic, etc.), use it to evaluate
# Otherwise, create a simple rule-based evaluator

def llm_judge_evaluate(prompt, response_a, response_b):
    """
    Compare two responses and determine which is better.
    Returns: 'A', 'B', or 'tie'
    """
    # YOUR CODE HERE
    # Option 1: Use API to ask which response is better
    # Option 2: Create rule-based evaluation (length, coherence, etc.)
    pass

# Test the evaluator
# YOUR CODE HERE

### 3.4 Pairwise Comparison

In [ ]:
# TODO: Conduct systematic pairwise comparison
comparison_prompts = [
    "<|user|>\nExplain recursion in programming.</s>\n<|assistant|>\n",
    "<|user|>\nWhat are the benefits of renewable energy?</s>\n<|assistant|>\n",
    "<|user|>\nHow does the internet work?</s>\n<|assistant|>\n",
    # Add more
]

results = []

for prompt in comparison_prompts:
    # Generate from both models
    # Evaluate which is better
    # Record results
    pass

# Summary statistics
# YOUR CODE HERE

### Questions:

**Q6:** Which evaluation metric do you find most useful for comparing generative models? Why?

*Your answer here*

**Q7:** What are the limitations of automatic evaluation metrics like ROUGE?

*Your answer here*

**Q8:** How would you design a comprehensive evaluation suite for a production chatbot?

*Your answer here*

---

## Bonus Challenge: Multi-Turn Conversations

Extend your model to handle multi-turn conversations.

**Your task:**
1. Create a multi-turn conversation dataset
2. Fine-tune your model on multi-turn data
3. Implement a conversation loop
4. Test coherence across multiple turns

In [ ]:
# TODO: Implement multi-turn conversation handling
def multi_turn_chat(model, tokenizer, max_turns=5):
    """
    Interactive multi-turn chat with the model.
    """
    conversation_history = []
    
    for turn in range(max_turns):
        # Get user input
        # Build conversation context
        # Generate response
        # Update history
        pass

# Test multi-turn conversation
# YOUR CODE HERE

---

## Final Reflection

**Q9:** Summarize what you learned about the two-stage fine-tuning approach (SFT + DPO).

*Your answer here*

**Q10:** What are the practical considerations for deploying a fine-tuned model in production?

*Your answer here*

**Q11:** How would you improve your fine-tuning pipeline if you had more resources (data, compute, time)?

*Your answer here*